# India Recession Predictor — Extended Model (Jan 2000 – Feb 2026)Trains a RandomForest recession classifier on **302 months** of data, covering **6 recession episodes**:1. Sep 2001 – Mar 2002 (dot-com + 9/11)2. Oct 2008 – Mar 2009 (Global Financial Crisis)3. Jun 2013 – Sep 2013 (Taper tantrum)4. Nov 2016 – Feb 2017 (Demonetization)5. Jul 2019 – Nov 2019 (Pre-COVID slowdown)6. Mar 2020 – Oct 2020 (COVID-19)**Output goes straight into the backend folder** — no manual copying needed afterward:- `backend/data/extended_2000_2026.csv`- `backend/models/rf_extended_2000.pkl`- `backend/models/scaler_extended_2000.pkl`- `backend/models/features_extended_2000.pkl`Run this notebook from inside `C:\final project\backend\` (or update `BACKEND_DIR` below).

In [ ]:
# CELL 1 — Install required packagesimport sys, subprocesssubprocess.run([sys.executable, "-m", "pip", "install",                "yfinance", "pandas-datareader", "openpyxl",                "imbalanced-learn", "scikit-learn", "--quiet"])print("Libraries ready")

In [ ]:
# CELL 2 — Paths# Point this at your backend folder. Everything this notebook produces# is saved directly into backend/data and backend/models.import osfrom pathlib import PathBACKEND_DIR = Path(r"C:\final project\backend")DATA_DIR    = BACKEND_DIR / "data"MODEL_DIR   = BACKEND_DIR / "models"DATA_DIR.mkdir(parents=True, exist_ok=True)MODEL_DIR.mkdir(parents=True, exist_ok=True)print(f"Data will be saved to : {DATA_DIR}")print(f"Model will be saved to: {MODEL_DIR}")

In [ ]:
# CELL 3 — Download market data, Jan 2000 - present# INR/USD, India VIX, Brent oil, Nifty 50 — all available back to 2000# (unlike IIP/CPI/GDP which only exist reliably from ~2011-2012)import pandas as pdimport yfinance as yfSTART = "2000-01-01"END   = "2026-03-01"tickers = {    "USDINR=X"  : "usdinr",    "^INDIAVIX" : "india_vix",    "BZ=F"      : "brent_oil",    "^NSEI"     : "nifty50",}print("Downloading market data from 2000...")series_list = []for ticker, name in tickers.items():    try:        df_t = yf.download(ticker, start=START, end=END,                            interval="1mo", auto_adjust=True, progress=False)        if len(df_t) > 0:            if isinstance(df_t.columns, pd.MultiIndex):                df_t.columns = df_t.columns.get_level_values(0)            s = df_t["Close"].resample("MS").last()            s.index = pd.to_datetime(s.index)            s.name = name            series_list.append(s)            print(f"  {name:<15} -> {len(s)} months")        else:            print(f"  {name} -> no data returned")    except Exception as e:        print(f"  {name} -> failed: {str(e)[:60]}")market = pd.concat(series_list, axis=1)market.index.name = "date"market = market.reset_index()market["date"] = pd.to_datetime(market["date"])print(f"\nMarket data shape: {market.shape}")market.head()

In [ ]:
# CELL 4 — Recession labels, Jan 2000 - Feb 2026# Six episodes, sourced from RBI Annual Reports / NIPFP Business Cycle# Dating / IMF Article IV consultations (see PRD section 1.2)RECESSION_PERIODS = [    ("2001-09-01", "2002-03-01"),  # Dot-com + 9/11    ("2008-10-01", "2009-03-01"),  # Global Financial Crisis    ("2013-06-01", "2013-09-01"),  # Taper tantrum    ("2016-11-01", "2017-02-01"),  # Demonetization    ("2019-07-01", "2019-11-01"),  # Pre-COVID slowdown    ("2020-03-01", "2020-10-01"),  # COVID-19]date_range = pd.date_range(start="2000-01-01", end="2026-02-01", freq="MS")labels = pd.DataFrame({"date": date_range, "recession_label": 0})for start, end in RECESSION_PERIODS:    mask = (labels["date"] >= start) & (labels["date"] <= end)    labels.loc[mask, "recession_label"] = 1total_months    = len(labels)recession_months = labels["recession_label"].sum()print(f"Total months     : {total_months}")print(f"Recession months : {recession_months}")print(f"Recession rate   : {recession_months/total_months:.1%}")print()for start, end in RECESSION_PERIODS:    mask = (labels["date"] >= start) & (labels["date"] <= end)    print(f"  {start[:7]} to {end[:7]} -> {mask.sum()} months")

In [ ]:
# CELL 5 — Feature engineeringdf = labels.merge(market, on="date", how="left")if "usdinr" in df.columns:    df["inr_level"]           = df["usdinr"]    df["inr_depreciation_1m"] = df["usdinr"].pct_change(1, fill_method=None) * 100    df["inr_depreciation_3m"] = df["usdinr"].pct_change(3, fill_method=None) * 100    print("INR features added")if "india_vix" in df.columns:    df["vix_level"]  = df["india_vix"]    df["vix_change"] = df["india_vix"].diff()    df["vix_3m_avg"] = df["india_vix"].rolling(3).mean()    print("VIX features added")if "brent_oil" in df.columns:    df["oil_price_usd"] = df["brent_oil"]    df["oil_change_3m"] = df["brent_oil"].pct_change(3, fill_method=None) * 100    print("Oil features added")if "nifty50" in df.columns:    df["nifty_return_1m"] = df["nifty50"].pct_change(1, fill_method=None) * 100    df["nifty_return_3m"] = df["nifty50"].pct_change(3, fill_method=None) * 100    df["nifty_3m_vol"]    = df["nifty50"].pct_change(1, fill_method=None).rolling(3).std() * 100    print("Nifty features added")# Composite market-stress score (z-scored average of key stress signals)stress_cols = [c for c in ["vix_level", "inr_depreciation_3m", "nifty_return_3m"] if c in df.columns]if stress_cols:    from sklearn.preprocessing import StandardScaler as SC    z = SC().fit_transform(df[stress_cols].ffill().fillna(0))    df["market_stress"] = z.mean(axis=1)    print("Market stress composite added")df = df.ffill().bfill()out_path = DATA_DIR / "extended_2000_2026.csv"df.to_csv(out_path, index=False)print(f"\nFinal shape: {df.shape}")print(f"Saved to: {out_path}")

In [ ]:
# CELL 6 — Train RandomForest on the 302-month datasetimport numpy as npfrom sklearn.preprocessing import StandardScalerfrom sklearn.ensemble import RandomForestClassifierfrom imblearn.over_sampling import SMOTEFEATURES = [c for c in [    "inr_level", "inr_depreciation_1m", "inr_depreciation_3m",    "vix_level", "vix_change", "vix_3m_avg",    "oil_price_usd", "oil_change_3m",    "nifty_return_1m", "nifty_return_3m", "nifty_3m_vol",    "market_stress",] if c in df.columns]TARGET = "recession_label"df_clean = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)X = df_clean[FEATURES].valuesy = df_clean[TARGET].valuesprint(f"Dataset  : {len(df_clean)} months")print(f"Recession: {y.sum()} months ({y.mean():.1%})")# 70/30 chronological split (per TRD: SPLIT=211 train months)SPLIT = int(len(y) * 0.70)scaler = StandardScaler()X_scaled = scaler.fit_transform(X)X_train, y_train = X_scaled[:SPLIT], y[:SPLIT]print(f"Train: {SPLIT} months (recession: {y_train.sum()})")print(f"Test : {len(y) - SPLIT} months")smote = SMOTE(random_state=42)X_resampled, y_resampled = smote.fit_resample(X_train, y_train)model = RandomForestClassifier(    n_estimators=400, max_depth=6,    class_weight="balanced", random_state=42)model.fit(X_resampled, y_resampled)probs = model.predict_proba(X_scaled)[:, 1]rec_idx = np.where(y == 1)[0]caught = (probs[rec_idx] >= 0.40).sum()avg_prob = probs[rec_idx].mean()print(f"\nRecession months caught: {caught}/{len(rec_idx)}")print(f"Avg P during recession : {avg_prob:.3f}")print()print("Feature importance:")for i in np.argsort(model.feature_importances_)[::-1]:    bar = chr(9608) * int(model.feature_importances_[i] * 200)    print(f"  {FEATURES[i]:<24} {model.feature_importances_[i]:.3f} {bar}")

In [ ]:
# CELL 7 — Save model, scaler, and feature list into backend/modelsimport picklepickle.dump(model,    open(MODEL_DIR / "rf_extended_2000.pkl", "wb"))pickle.dump(scaler,   open(MODEL_DIR / "scaler_extended_2000.pkl", "wb"))pickle.dump(FEATURES, open(MODEL_DIR / "features_extended_2000.pkl", "wb"))print("Saved:")print(f"  {MODEL_DIR / 'rf_extended_2000.pkl'}")print(f"  {MODEL_DIR / 'scaler_extended_2000.pkl'}")print(f"  {MODEL_DIR / 'features_extended_2000.pkl'}")print()print("Dataset saved at:")print(f"  {DATA_DIR / 'extended_2000_2026.csv'}")print()print("Next: restart your Flask backend (python app.py) and check")print("http://localhost:5000/api/v1/health — the extended model")print("will now load automatically (it's first in config.py's search order).")